In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
# Keep text editable in Adobe Illustrator SVG/PDF outputs
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

from matplotlib.patches import Patch

# path setting
BASE_PATH = Path.cwd().parent.parent

RAW = BASE_PATH / 'Data' / 'raw'
TEMP = BASE_PATH / 'Data' / 'temp'
USE = BASE_PATH / 'Data' / 'use'
FIGURES = BASE_PATH / 'Results' / 'Figures'
TABLES = BASE_PATH / 'Results' / 'Tables'

for path in [RAW, TEMP, USE, FIGURES, TABLES]:
    path.mkdir(parents=True, exist_ok=True)

print(f"✅ BASE_PATH: {BASE_PATH}")

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams.update({'font.size': 20})


def find_raw_file(filename):
    direct_path = RAW / filename
    if direct_path.exists():
        return direct_path

    matches = sorted(RAW.rglob(filename))
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise FileExistsError(
            f"Multiple files named {filename} found under {RAW}: {matches}"
        )
    raise FileNotFoundError(
        f"Could not find {filename} in {RAW} or its subdirectories."
    )


In [ ]:
scenario_path = find_raw_file('AR6_Scenarios_Database_World_v1.1.csv')
metadata_path = find_raw_file('AR6_Scenarios_Database_metadata_indicators_v1.1.xlsx')

A = pd.read_csv(scenario_path)
B = pd.read_excel(metadata_path, sheet_name='meta_Ch3vetted_withclimate')

print(f"Loaded scenario database: {scenario_path}")
print(f"Loaded metadata: {metadata_path}")
print(f"Scenario data shape: {A.shape}")
print(f"Metadata shape: {B.shape}")


In [43]:
# =========================
# 1. Select Variable
# =========================
var_list = [
    "Capacity|Electricity|Coal",
    "Capacity|Electricity|Coal|w/ CCS",
    "Capacity|Electricity|Coal|w/o CCS",
    "Secondary Energy|Electricity|Coal",
    "Secondary Energy|Electricity|Coal|w/ CCS",
    "Secondary Energy|Electricity|Coal|w/o CCS"
]

# =========================
# 2. Scenario merge with Category
# =========================
merged = A.merge(
    B[['Model', 'Scenario', 'Category']],
    on=['Model', 'Scenario'],
    how='inner'
)

result = merged[merged['Variable'].isin(var_list)].copy()
result = result.sort_values(['Model', 'Scenario', 'Variable'])

In [44]:
# Select target variables
target_vars = [
    "Capacity|Electricity|Coal|w/o CCS",
    "Secondary Energy|Electricity|Coal|w/o CCS"
]
df = result[result['Variable'].isin(target_vars)].copy()

# Check completeness (both variables must exist)
valid_keys = (
    df.groupby(['Model', 'Scenario', 'Region'])['Variable']
      .nunique()
      .reset_index(name='var_count')
)

# Keep only groups that have both variables
valid_keys = valid_keys[valid_keys['var_count'] == len(target_vars)]

# 4. Filter back to original data
result1 = df.merge(
    valid_keys[['Model', 'Scenario', 'Region']],
    on=['Model', 'Scenario', 'Region'],
    how='inner'
)

# 5. Sort the results
result1 = result1.sort_values(
    ['Model', 'Scenario', 'Region', 'Variable']
)

In [45]:
data = result1.copy()
data['Variable'] = data['Variable'].str.strip()

# CF mapping

cf_map = {
    "Coal": (
        "Capacity|Electricity|Coal",
        "Secondary Energy|Electricity|Coal"
    ),
    "Coal|w/ CCS": (
        "Capacity|Electricity|Coal|w/ CCS",
        "Secondary Energy|Electricity|Coal|w/ CCS"
    ),
    "Coal|w/o CCS": (
        "Capacity|Electricity|Coal|w/o CCS",
        "Secondary Energy|Electricity|Coal|w/o CCS"
    )
}

conversion = 31.70979198

years = [c for c in data.columns if str(c).isdigit()]

result_list = []

# Year-by-year calculation
for y in years:

    tmp = data[['Model','Scenario','Region','Category','Variable', y]].copy()
    tmp = tmp.rename(columns={y: 'Value'})

    wide = tmp.pivot_table(
        index=['Model','Scenario','Region','Category'],
        columns='Variable',
        values='Value',
        aggfunc='first'
    ).reset_index()

    for name, (cap, gen) in cf_map.items():

        # Skip if required columns are missing
        if cap not in wide.columns or gen not in wide.columns:
            continue

        cap_vals = wide[cap]
        gen_vals = wide[gen]

        # CF is set to NaN when capacity < 1
        cf = np.where(
            (cap_vals < 1) | (cap_vals == 0) | (pd.isna(cap_vals)) | (gen_vals == 0) ,
            np.nan,
            gen_vals / cap_vals * conversion
        )

        out = wide[['Model','Scenario','Region','Category']].copy()
        out[f'CF|{name}'] = cf
        out['Year'] = y

        result_list.append(out)

# Concatenate all yearly results
result2 = pd.concat(result_list, ignore_index=True)
result2 = result2.sort_values(
    ['Model','Scenario','Region','Category','Year']
).reset_index(drop=True)

In [46]:
# 1. Identify abnormal CF values greater than 1
bad_keys = (
    result2[result2['CF|Coal|w/o CCS'] > 1]
    [['Model', 'Scenario']]
    .drop_duplicates()
)

# 2. Mark and remove these model-scenario combinations
result3 = result2.merge(
    bad_keys.assign(bad=1),
    on=['Model', 'Scenario'],
    how='left'
)

result3 = result3[result3['bad'].isna()].drop(columns=['bad'])

In [ ]:
# For some scenarios, the years range from 2005 to 2100 with values every 5 years.
# If a value is missing, interpolate using the mean of the two neighboring values.
# Construct complete timeline (2005–2100, every 5 years)
years_full = list(range(2005, 2101, 5))

# Interpolation function (strict inside-only version)
def interpolate_group(df):
    df = df.copy()

    df['Year'] = df['Year'].astype(int)
    df = df.set_index('Year')

    df = df.reindex(years_full)

    for col in ['Model','Scenario','Region','Category']:
        df[col] = df[col].ffill().bfill()

    cf_cols = [c for c in df.columns if c.startswith('CF|')]

    for c in cf_cols:
        s = df[c]

         # interpolate only within real values (no extrapolation)
        if s.notna().sum() < 2:
            continue

        df[c] = s.interpolate(
            method='linear',
            limit_area='inside'   # ⭐关键：禁止外推（只填内部缺口）
        )

    return df.reset_index()


# Apply interpolation by group

result4 = (
    result3
    .groupby(['Model','Scenario','Region','Category'], group_keys=False)
    .apply(interpolate_group)
)

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Nature-style capacity-factor trend plot with smoothed IQR bands

cf_plot_df = result4.copy()

def group_map(c):
    if c in ['C1', 'C2', 'C3', 'C4']:
        return 'C1-C4'
    elif c in ['C5', 'C6']:
        return 'C5-C6'
    elif c in ['C7', 'C8']:
        return 'C7-C8'
    else:
        return np.nan

cf_plot_df['Group'] = cf_plot_df['Category'].apply(group_map)
cf_plot_df = cf_plot_df.dropna(subset=['Group'])
cf_plot_df['Year'] = cf_plot_df['Year'].astype(int)
cf_plot_df = cf_plot_df[cf_plot_df['Year'] >= 2005].copy()

groups_3 = ['C1-C4', 'C5-C6', 'C7-C8']

labels = {
    'C1-C4': 'Limit to 2°C',
    'C5-C6': 'Limit to 3°C',
    'C7-C8': 'Exceed 3°C'
}

colors = {
    'C1-C4': '#1A05A2',
    'C5-C6': '#8F0177',
    'C7-C8': '#DE1A58'
}

records = []
for g in groups_3:
    sub = cf_plot_df[cf_plot_df['Group'] == g]
    for y in sorted(sub['Year'].unique()):
        vals = sub.loc[sub['Year'] == y, 'CF|Coal|w/o CCS'].dropna().values
        if len(vals) < 2:
            continue
        records.append({
            'Group': g,
            'Year': int(y),
            'mean': np.mean(vals),
            'p25': np.percentile(vals, 25),
            'p75': np.percentile(vals, 75),
            'n': len(vals)
        })

summary = pd.DataFrame(records)

def smooth_series(s, window=3):
    return (
        pd.Series(s)
        .rolling(window=window, center=True, min_periods=1)
        .mean()
        .to_numpy()
    )

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 7.2,
    'axes.linewidth': 0.6,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'xtick.major.size': 3,
    'ytick.major.size': 3,
})

fig, ax = plt.subplots(figsize=(5.4, 3.0))

# Smoothed IQR bands
for g in groups_3:
    tmp = summary[summary['Group'] == g].sort_values('Year')

    x = tmp['Year'].to_numpy()
    p25 = smooth_series(tmp['p25'].to_numpy(), window=3)
    p75 = smooth_series(tmp['p75'].to_numpy(), window=3)

    ax.fill_between(
        x,
        p25,
        p75,
        color=colors[g],
        alpha=0.055,
        linewidth=0,
        zorder=1
    )

    ax.plot(
        x,
        p25,
        color=colors[g],
        linewidth=0.45,
        alpha=0.24,
        zorder=1.5
    )

    ax.plot(
        x,
        p75,
        color=colors[g],
        linewidth=0.45,
        alpha=0.24,
        zorder=1.5
    )

# Smoothed mean lines
for g in groups_3:
    tmp = summary[summary['Group'] == g].sort_values('Year')

    x = tmp['Year'].to_numpy()
    mean = smooth_series(tmp['mean'].to_numpy(), window=3)

    ax.plot(
        x,
        mean,
        color=colors[g],
        linewidth=1.65,
        zorder=3
    )

# 2024 reference line
ax.axvline(
    2024,
    linestyle=(0, (3, 2)),
    color='0.55',
    linewidth=0.7,
    zorder=0
)

ax.text(
    2025,
    0.035,
    '2024',
    fontsize=7.2,
    color='0.35',
    ha='left',
    va='bottom'
)

ax.set_xlim(2005, 2100)
ax.set_ylim(0, 0.82)
ax.set_xticks([2010, 2030, 2050, 2070, 2090])
ax.set_yticks(np.arange(0, 0.81, 0.2))

ax.set_xlabel('Year')
ax.set_ylabel('Coal capacity factor')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(False)
ax.tick_params(direction='out')

line_handles = [
    Line2D(
        [0],
        [0],
        color=colors[g],
        linewidth=1.65,
        label=labels[g]
    )
    for g in groups_3
]

band_handle = Patch(
    facecolor='0.70',
    edgecolor='0.55',
    linewidth=0.5,
    alpha=0.16,
    label='25-75th percentile'
)

ax.legend(
    handles=line_handles + [band_handle],
    loc='center left',
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    handlelength=2.0,
    labelspacing=0.55,
    borderaxespad=0
)

fig.tight_layout(pad=0.8)

fig_svg = FIGURES / 'CF_trends_nature_style.svg'

fig.savefig(fig_svg, bbox_inches='tight')

print(f"Saved: {fig_svg}")

plt.show()

In [ ]:
mean_df = summary.pivot(index='Year', columns='Group', values='mean')[groups_3]

mean_df = mean_df.rename(columns={
    'C1-C4': 'Limit to 2°C',
    'C5-C6': 'Limit to 3°C',
    'C7-C8': 'Exceed 3°C'
})

out_csv = TABLES / 'CF_trend.csv'
mean_df.to_csv(out_csv)
print(f"Saved: {out_csv}")

mean_df